In [ ]:

# use spark session if needed
from kpi_distribution_utils import analyze_kpi
from utils.consts import SHARED_DIR_PATH
from utils.utils import SparkDataManager

sdm = SparkDataManager()

PREPROCESSED_DATASET_PATH = SHARED_DIR_PATH / "preprocessed_dataset"

In [ ]:
pm_df = sdm.read_parquet(PREPROCESSED_DATASET_PATH / "pm_data_long")

## Some thougths for KPI distributions:
- kpis have very different distributions, with occuring regime shifts and for some kpis, frequent periods with NO DATA or long-term periods with NO DATA
- kpis might be fully constant
- there are kpis with very little coverage, and big spikes from time to time, what is the source?

## Solutions
- kpis with different distributions shouldnt be naively standardized using z-score globally, but they should be standardized using robust standarization methods, combined with transformations fe. log1p, Box-Cox, Yeo-Johnson (exp. distributions). Robust methods should be prepared for regime change, but still include them after standardization.
- FE: Rolling window MinMax Scaler, rolling MAD or IQR

- No data long periods - periods where little bits of data are available at the start of a KPI, but then there is a long period of nulls, should be dropped and new starting time point should be set, where the data is truly (kinda) continous with minimal imputing

- Maybe we should assign NULL periods to a binary Dataframe? To model NULL distributions aswell?

- constant kpis should be flagged as consts, and returned as they are
- very little changing kpis, should be modeled as is, as they do sometimes have big spikes, and appear more like exp. distribution kpis

In [ ]:
kpis_list = pm_df.select("kpi_id").distinct().rdd.flatMap(lambda x: x).collect()
print(*kpis_list, sep="\n")

In [ ]:
for kpi in kpis_list[:100]:
    analyze_kpi(pm_df, kpi_id=kpi)